In [1]:
import duckdb
from pathlib import Path

In [2]:
BASE = Path.cwd().parent.parent
DATA_DIR = BASE / "data" / "options"
OUT_DIR = BASE / "output"

OHLC_GLOB   = str(DATA_DIR / "ohlc" / "*.parquet")

In [3]:
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

**On and off exchange trades**
- Code=58 then on-exchange trades
- Code=85 then off-exchange trades

We do not need the off-exchange trades, since the prices are different from the regular market prices

In [16]:
duckdb.sql(f"""
    SELECT 
        publisher_id,
        COUNT(*) as count
    FROM read_parquet('{OHLC_GLOB}')
    WHERE symbol LIKE '%_OMC%'
       OR symbol LIKE '%_OMP%'
    GROUP BY publisher_id
""").df()

,publisher_id,count
0,58,340313
1,85,25579


Totally $340,313$ on-exchange trades spanning from 2023-03-20 to 2026-06-26

In [7]:
duckdb.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(ts_event) as non_null,
        MIN(ts_event) as earliest,
        MAX(ts_event) as latest
    FROM read_parquet('{OHLC_GLOB}')
    WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
    AND publisher_id = 58
""").df()

,total_rows,non_null,earliest,latest
0,340313,340313,2023-03-20 12:06:00+02:00,2026-06-26 18:50:00+03:00


Only $4,327$ contracts were traded

In [17]:
duckdb.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT instrument_id) as unique_contracts
    FROM read_parquet('{OHLC_GLOB}')
    WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
    AND publisher_id = 58
""").df()

,total_rows,unique_contracts
0,340313,4327
